# pycycle — RR Lyrae Template Fitting

This notebook demonstrates fitting RR Lyrae templates to a multiband light curve using two template libraries:

1. **rr-templates** (Long / Stringer et al. 2019) — averaged SDSS/DES templates with full physics parameterisation (distance modulus, dust, amplitude, phase).
2. **Multiband-templates** (Baeza-Villagra et al. 2025) — 136 RRab + 144 RRc individual DECam griz templates.

Both libraries must be cloned separately:
```bash
git clone https://github.com/longjp/rr-templates
git clone https://github.com/kbbaeza/Multiband-templates
```

**Algorithm:** grid search over test periods with Newton iterations to jointly optimise phase and linear parameters (amplitude, offset, dust). The inner loop is accelerated by a Cython extension when available.

In [ ]:
import importlib.resources
import numpy as np
import matplotlib.pyplot as plt

import pycycle
from pycycle.templates import (
    load_rr_template,
    load_multiband_templates,
    average_multiband_templates,
)
from pycycle.template_fit import TemplateFitter

print('pycycle version:', pycycle.__version__)

## 0. Set template paths

Edit these to point at your local clones.

In [ ]:
import os

RR_TEMPLATES_SDSS = os.path.expanduser('~/software/rr-templates/template_sdss')
RR_TEMPLATES_DES  = os.path.expanduser('~/software/rr-templates/template_des')
MB_RRAB_ZIP       = os.path.expanduser('~/software/Multiband-templates/RRab_normalized.zip')
MB_RRC_ZIP        = os.path.expanduser('~/software/Multiband-templates/RRc_normalized.zip')

## 1. Load the sample light curve

We use the bundled B1392 SDSS Stripe 82 RR Lyrae dataset (5 bands: u g r i z).

In [ ]:
data_path = importlib.resources.files('pycycle.data').joinpath('B1392all.tab')
hjd, mag, magerr, filts = np.loadtxt(str(data_path), unpack=True)

# standard quality cut
ok = (magerr >= 0.0) & (magerr <= 0.2)
hjd, mag, magerr, filts = hjd[ok], mag[ok], magerr[ok], filts[ok]

filtnams = ['u', 'g', 'r', 'i', 'z']
print(f'{len(hjd)} observations across {len(filtnams)} bands')

In [ ]:
# quick look — r-band raw light curve
r_ok = filts == 2
hjd0 = int(hjd.min())
fig, ax = plt.subplots(figsize=(10, 3))
ax.errorbar(hjd[r_ok] - hjd0, mag[r_ok], yerr=magerr[r_ok],
            fmt='o', ms=4, alpha=0.7, color='steelblue')
ax.invert_yaxis()
ax.set_xlabel(f'HJD − {hjd0} [days]')
ax.set_ylabel('r magnitude')
ax.set_title('B1392 — r-band raw light curve')
plt.tight_layout(); plt.show()

## 2. rr-templates fit (SDSS)

**Model:** $m_b(t) = \beta_b(\omega) + \mu + E(B{-}V)\cdot R_b + A\cdot\gamma_b(\omega t + \phi)$

Free parameters: $\mu$ (distance modulus), $E(B{-}V)$ (dust), $A$ (amplitude), $\phi$ (phase). The $\beta_b(\omega)$ term is a frequency-dependent absolute magnitude correction pre-computed in the template.

In [ ]:
template_sdss = load_rr_template(RR_TEMPLATES_SDSS, name='sdss')
print(template_sdss)
print('Bands:', template_sdss.bands)
print('Dust coefficients:', dict(zip(template_sdss.bands, template_sdss.dust)))

In [ ]:
# quick look at the template shape
fig, ax = plt.subplots(figsize=(8, 3))
colors = ['purple', 'seagreen', 'steelblue', 'orange', 'red']
for i, (band, c) in enumerate(zip(template_sdss.bands, colors)):
    ax.plot(template_sdss.phase, template_sdss.gamma[i], label=band, color=c)
ax.set_xlabel('Phase')
ax.set_ylabel('Template value')
ax.set_title('rr-templates SDSS — normalised template per band')
ax.legend(ncol=5)
plt.tight_layout(); plt.show()

In [ ]:
# use only the 3 bands present in both the SDSS template and our data
# (the template has g, i, r, u, z — all 5 match)
fitter_rr = TemplateFitter(template_sdss, n_newton=5)
result_rr = fitter_rr.fit(hjd, mag, magerr, filts, filtnams,
                           pmin=0.2, dphi=0.02, pmax=1.0)

print(f'\nBest period (rr-templates SDSS): {result_rr.best_period:.7f} days')
print('Best coefficients:', {k: f'{v:.4f}' for k, v in result_rr.best_coeffs.items()})

In [ ]:
result_rr.plot_rss()
plt.show()

In [ ]:
result_rr.plot_phased()
plt.show()

## 3. rr-templates fit (DES)

The DES template covers g, i, r, Y, z bands.  We fit only the g, r, i, z overlap with our data.

In [ ]:
template_des = load_rr_template(RR_TEMPLATES_DES, name='des')
print(template_des)

# keep only observations in bands that exist in the DES template
des_bands = template_des.bands  # ['g', 'i', 'r', 'Y', 'z']
# filtnams = ['u','g','r','i','z'] — keep g(1), r(2), i(3), z(4)
keep_codes = [fi for fi, fn in enumerate(filtnams) if fn in des_bands]
mask_des = np.isin(filts.astype(int), keep_codes)
# remap filter codes so they index into filtnams_des
filtnams_des = [fn for fn in filtnams if fn in des_bands]
remap = {old: filtnams_des.index(filtnams[old]) for old in keep_codes}
filts_des = np.array([remap[int(f)] for f in filts[mask_des]])

fitter_des = TemplateFitter(template_des, n_newton=5)
result_des = fitter_des.fit(hjd[mask_des], mag[mask_des], magerr[mask_des],
                             filts_des, filtnams_des, pmin=0.2, dphi=0.02, pmax=1.0)

print(f'\nBest period (rr-templates DES): {result_des.best_period:.7f} days')
print('Best coefficients:', {k: f'{v:.4f}' for k, v in result_des.best_coeffs.items()})

## 4. Multiband-templates fit (RRab)

**Model:** $m_b(t) = \mu_b + A\cdot\gamma_b(\omega t + \phi)$

We average the 136 RRab DECam templates to get a single representative template, then fit it. The Multiband templates cover g, i, r, z (griz).

In [ ]:
print('Loading RRab templates (this takes a few seconds)...')
rrab_templates = load_multiband_templates(MB_RRAB_ZIP, n_phase=100)
print(f'Loaded {len(rrab_templates)} RRab templates, bands: {rrab_templates[0].bands}')

In [ ]:
# view a few individual templates
fig, axes = plt.subplots(1, 4, figsize=(12, 3), sharey=False)
for i, (ax, band) in enumerate(zip(axes, rrab_templates[0].bands)):
    for t in rrab_templates[:20]:
        ax.plot(t.phase, t.gamma[i], alpha=0.3, lw=0.8, color='steelblue')
    ax.set_title(band); ax.set_xlabel('Phase')
axes[0].set_ylabel('Normalised magnitude')
plt.suptitle('First 20 RRab templates (Multiband-templates)')
plt.tight_layout(); plt.show()

In [ ]:
avg_rrab = average_multiband_templates(rrab_templates)
print(avg_rrab)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, (ax, band) in enumerate(zip(axes, avg_rrab.bands)):
    ax.plot(avg_rrab.phase, avg_rrab.gamma[i], color='k', lw=1.5)
    ax.set_title(band); ax.set_xlabel('Phase')
axes[0].set_ylabel('Normalised magnitude')
plt.suptitle('Averaged RRab template')
plt.tight_layout(); plt.show()

In [ ]:
# keep only griz observations (Multiband bands)
mb_bands = avg_rrab.bands  # ['g', 'i', 'r', 'z']
keep_mb = [fi for fi, fn in enumerate(filtnams) if fn in mb_bands]
mask_mb = np.isin(filts.astype(int), keep_mb)
filtnams_mb = [fn for fn in filtnams if fn in mb_bands]
remap_mb = {old: filtnams_mb.index(filtnams[old]) for old in keep_mb}
filts_mb = np.array([remap_mb[int(f)] for f in filts[mask_mb]])

fitter_mb = TemplateFitter(avg_rrab, n_newton=5)
result_mb = fitter_mb.fit(hjd[mask_mb], mag[mask_mb], magerr[mask_mb],
                           filts_mb, filtnams_mb, pmin=0.2, dphi=0.02, pmax=1.0)

print(f'\nBest period (Multiband RRab avg): {result_mb.best_period:.7f} days')
print('Best coefficients:', {k: f'{v:.4f}' for k, v in result_mb.best_coeffs.items()})

In [ ]:
result_mb.plot_rss()
plt.show()

In [ ]:
result_mb.plot_phased()
plt.show()

## 5. Summary comparison

In [ ]:
gold = 0.5016247
print(f'Gold period (OGLE):              {gold:.7f} days')
print(f'rr-templates SDSS best period:   {result_rr.best_period:.7f} days')
print(f'rr-templates DES  best period:   {result_des.best_period:.7f} days')
print(f'Multiband RRab avg best period:  {result_mb.best_period:.7f} days')

In [ ]:
# side-by-side RSS comparison (rr-templates SDSS vs Multiband)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))

ax1.plot(result_rr.periods, result_rr.rss / result_rr.rss.max(),
         color='steelblue', lw=0.7, label='rr-templates SDSS')
ax1.axvline(result_rr.best_period, color='k', lw=1, ls='--')
ax1.set_xlabel('Period [days]'); ax1.set_ylabel('Normalised RSS')
ax1.set_title('rr-templates (SDSS)')

ax2.plot(result_mb.periods, result_mb.rss / result_mb.rss.max(),
         color='seagreen', lw=0.7, label='Multiband RRab')
ax2.axvline(result_mb.best_period, color='k', lw=1, ls='--')
ax2.set_xlabel('Period [days]'); ax2.set_ylabel('Normalised RSS')
ax2.set_title('Multiband-templates (RRab avg)')

plt.tight_layout(); plt.show()

## 6. Fitting against individual Multiband templates

Instead of the average, fit against each RRab template independently and pick the one with the lowest RSS.

In [ ]:
best_rss = np.inf
best_result_ind = None
best_template_name = None

for tmpl in rrab_templates[:30]:  # use first 30 for speed; remove [:30] for all
    fitter_i = TemplateFitter(tmpl, n_newton=3)
    res_i = fitter_i.fit(hjd[mask_mb], mag[mask_mb], magerr[mask_mb],
                          filts_mb, filtnams_mb, pmin=0.2, dphi=0.02, pmax=1.0)
    if res_i.rss.min() < best_rss:
        best_rss = res_i.rss.min()
        best_result_ind = res_i
        best_template_name = tmpl.name

print(f'Best individual template: {best_template_name}')
print(f'Best period: {best_result_ind.best_period:.7f} days')

In [ ]:
best_result_ind.plot_phased()
plt.show()